# 9.4 Model selection with correlational data
In this notebook we show how to use modular structure as model selection criterion when thresholding a correlation network. We do this using the codelength savings (description length compression) $\frac{L(1)-L(M)}{L(1)}$, where $L(1)$ is the codelength with a 1-level (uncompressed) partition and $L(M)$ is the maximally compressed codelength using the optimal partition $M$. We crossvalidate the threshold $\tau$ and evaluate each threshold (model) through the codelength savings in the test network given the model (partition) based on the training data, and choose the best threshold $\tau^*$ such that $\tau^*=\mathrm{argmax}_\tau\frac{L^{\tau,test}(1)-L^{\tau,test}(M^{\tau,train})}{L^{\tau,test}(1)}$. This threshold gives the best balance between over- and underfitting modular structure to data.

In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import infomap
from sklearn.metrics import adjusted_mutual_info_score

In [ ]:
# Blend a color toward white by `factor` to get a lighter shade
def lighten(color, factor=0.5):
    return tuple(v + (1 - v) * factor for v in color[:3])


# For each edge pick one endpoint's module color at random, then lighten it
def make_edge_colors(G, modules, palette, rng, factor=0.3):
    colors = []
    for u, v in G.edges():
        node = rng.choice([u, v])
        colors.append(lighten(palette[modules[node] - 1], factor))
    return colors

In [ ]:
def blockCov(num_blocks, block_size, covariance):
    # block diagonal matrix to use as covariance matrix in multivariate normal distribution
    block = np.full((block_size, block_size), covariance)
    np.fill_diagonal(block, 1)
    return np.block(
        [
            [
                block if i == j else np.zeros((block_size, block_size))
                for j in range(num_blocks)
            ]
            for i in range(num_blocks)
        ]
    )

In [ ]:
def codelength_savings(sample, thresholds):
    # crossvalidate the threshold to find the best balance between over and underfitting modular structure to data
    train_codelength_savings = []
    test_codelength_savings = []
    codelength_savings_best = 0
    sample_df = pd.DataFrame(data=sample)
    for threshold in thresholds:
        # 2-fold splitting of samples
        train_df = sample_df.sample(frac=0.5, axis=0)
        test_df = sample_df.drop(train_df.index)
        # make training network and run Infomap
        sample_train = train_df.to_numpy(copy=True)
        sample_test = test_df.to_numpy(copy=True)

        r_train = np.dot(sample_train.T, sample_train) / sample_train.shape[0]
        np.fill_diagonal(r_train, 0)
        G_train = nx.from_numpy_array(np.abs(r_train * (r_train > threshold)))
        # G_train = nx.from_numpy_array(r_train * (r_train > threshold))

        infomap_train = infomap.Infomap(silent=True, two_level=True)
        infomap_train.add_networkx_graph(G_train)
        infomap_train.run()
        train_codelength_savings.append(infomap_train.relative_codelength_savings)
        # make test network and run Infomap with the modules from the training network
        r_test = np.dot(sample_test.T, sample_test) / sample_test.shape[0]
        np.fill_diagonal(r_test, 0)
        G_test = nx.from_numpy_array(np.abs(r_test * (r_test > threshold)))
        # G_test = nx.from_numpy_array(r_test * (r_test > threshold))

        infomap_test = infomap.Infomap(silent=True, two_level=True, no_infomap=True)
        infomap_test.add_networkx_graph(G_test)
        infomap_test.run(initial_partition=dict(infomap_train.modules))
        test_codelength_savings.append(infomap_test.relative_codelength_savings)
        if infomap_test.relative_codelength_savings > codelength_savings_best:
            codelength_savings_best = infomap_test.relative_codelength_savings
    return train_codelength_savings, test_codelength_savings

In [ ]:
# sample multivariate data with planted modular structure
planted_cov = 0.3
nSamples = 100
nModules = 8
nNodesPerModule = 30

cov = blockCov(nModules, nNodesPerModule, planted_cov)
mean = np.zeros((nModules * nNodesPerModule,))
sample = np.random.multivariate_normal(mean, cov, size=nSamples)
plantedPartition = []
for i in range(1, nModules + 1):
    plantedPartition += np.full(nNodesPerModule, i).tolist()
# do repeated cross validation to find the threshold that gives
# the best balance between over and underfitting modular structure to data
thresholds = np.arange(0, 0.75, 0.05)
nRuns = 10
train_codelength_savings, test_codelength_savings = (
    np.zeros((1, len(thresholds))),
    np.zeros((1, len(thresholds))),
)
for _ in range(0, nRuns):
    train_codelength_savings_, test_codelength_savings_ = codelength_savings(
        sample, thresholds
    )
    train_codelength_savings += train_codelength_savings_
    test_codelength_savings += test_codelength_savings_
threshold_best = thresholds[test_codelength_savings.argmax()]
fig, ax = plt.subplots(1, 1, figsize=(3.5, 3.5))
ax.plot(thresholds, train_codelength_savings[0] / nRuns, label="Training")
ax.plot(thresholds, test_codelength_savings[0] / nRuns, label="Test")
ax.plot(
    threshold_best,
    test_codelength_savings.max() / nRuns,
    "o",
    color="darkorange",
    label="Best threshold",
)
ax.set_xlim(0.0, 0.72)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
(
    ax.legend(frameon=False),
    ax.set_xlabel("Threshold"),
    ax.set_ylabel("Codelength savings"),
);

In [ ]:
# Visualise the three thresholding regimes side by side:
#   left  – best threshold τ* selected by cross-validation
#   middle – underfit (low τ → dense graph, modules merge)
#   right  – overfit  (high τ → sparse graph, modules fragment)
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

rng = np.random.default_rng(42)

# Compute absolute correlation matrix; diagonal set to 0 to remove self-loops
r = np.abs(np.dot(sample.T, sample) / sample.shape[0])
np.fill_diagonal(r, 0)

# --- Best threshold (left panel) ---
threshold = threshold_best
G_best = nx.from_numpy_array(r * (r > threshold))
im = infomap.Infomap(silent=True, two_level=True)
im.add_networkx_graph(G_best)
im.run()
nodelist = G_best.nodes
modules = dict(im.modules)
num_modules = im.num_top_modules

# Build layout from connected nodes only; scatter isolates near the centre
# so the spring layout is not distorted by degree-0 nodes
connected = [n for n in G_best.nodes if G_best.degree(n) > 0]
# pos = nx.spring_layout(G_best.subgraph(connected), seed=42)
pos = nx.forceatlas2_layout(
    G_best.subgraph(connected),
    jitter_tolerance=1.0,
    scaling_ratio=0.2,
    gravity=1.5,
    strong_gravity=True,
    node_mass=None,
    node_size=None,
    weight=None,
    seed=42,
)
center = np.mean(list(pos.values()), axis=0)
for n in nx.isolates(G_best):
    pos[n] = center + rng.uniform(-0.1, 0.1, 2)

AMI = adjusted_mutual_info_score(list(modules.values()), plantedPartition)
palette = sns.color_palette("Set3", n_colors=num_modules)
node_colors = [palette[modules[node] - 1] for node in nodelist]
nx.draw(
    G_best,
    pos=pos,
    ax=ax1,
    node_color=node_colors,
    edgecolors="gray",
    linewidths=0.2,
    node_size=150,
    edge_color=make_edge_colors(G_best, modules, palette, rng),
    width=0.7,
)
ax1.axis("off")
ax1.set_title(
    f"Best threshold = {threshold_best:.2f}, {im.num_non_trivial_top_modules} non-trivial modules\nAMI = {AMI:.2f}"
)

# --- Underfit (middle panel): low threshold keeps weak correlations → modules merge ---
threshold = 0.1
G_underfit = nx.from_numpy_array(r * (r > threshold))
im = infomap.Infomap(silent=True, two_level=True)
im.add_networkx_graph(G_underfit)
im.run()
nodelist = G_underfit.nodes
modules = dict(im.modules)
num_modules = im.num_top_modules
AMI = adjusted_mutual_info_score(list(modules.values()), plantedPartition)
palette = sns.color_palette("Set3", n_colors=num_modules)
node_colors = [palette[modules[node] - 1] for node in nodelist]
nx.draw(
    G_underfit,
    pos=pos,
    ax=ax2,
    node_color=node_colors,
    edgecolors="gray",
    linewidths=0.2,
    node_size=150,
    edge_color=make_edge_colors(G_underfit, modules, palette, rng),
    width=0.3,
)
ax2.axis("off")
ax2.set_title(
    f"Underfit using threshold = {threshold:.2f}, {im.num_top_modules} modules\nAMI = {AMI:.2f}"
)

# --- Overfit (right panel): high threshold keeps only strong correlations → modules fragment ---
threshold = 0.5
G_overfit = nx.from_numpy_array(r * (r > threshold))
im = infomap.Infomap(silent=True, two_level=True)
im.add_networkx_graph(G_overfit)
im.run()
nodelist = G_overfit.nodes
modules = dict(im.modules)
num_modules = im.num_top_modules
AMI = adjusted_mutual_info_score(list(modules.values()), plantedPartition)
palette = sns.color_palette("Set3", n_colors=num_modules)
node_colors = [palette[modules[node] - 1] for node in nodelist]
nx.draw(
    G_overfit,
    pos=pos,
    ax=ax3,
    node_color=node_colors,
    edgecolors="gray",
    linewidths=0.2,
    node_size=150,
    edge_color=make_edge_colors(G_overfit, modules, palette, rng),
    width=0.7,
)
ax3.axis("off")
ax3.set_title(
    f"Overfit using threshold = {threshold:.2f}, {im.num_non_trivial_top_modules} non-trivial modules\nAMI = {AMI:.2f}"
);

## Regularized Map Equation on full data

Cross-validation uses only half the data at each step, splitting samples into training and test sets to estimate generalization. The Regularized Map Equation takes a different route: it applies a Dirichlet prior to the map equation's transition rates, regularizing against overfitting while using all available samples for inference.

For each threshold $\tau$, we threshold the full correlation matrix into an unweighted binary network and run the Regularized Map Equation. The method simultaneously determines the partition and the threshold by maximizing the description length compression

$$\Delta D(\tau) = \frac{D^1(\tau) - D^*(\tau)}{D^1(\tau)},$$

where $D^1(\tau)$ is the description length of the uncompressed one-cluster partition and $D^*(\tau)$ is the description length of the best-compressed partition at threshold $\tau$. The optimal threshold is $\tau^* = \operatorname{argmax}_\tau\, \Delta D(\tau)$.

Because the Dirichlet prior penalizes overfitting rather than requiring held-out data, the Regularized Map Equation recovers planted structure with fewer samples than cross-validation.

In [ ]:
# Full correlation matrix using all samples
r_full = np.dot(sample.T, sample) / sample.shape[0]
np.fill_diagonal(r_full, 0)

# Sweep thresholds: RegMapEq operates on unweighted (binary) networks
reg_codelength_savings = []
for threshold in thresholds:
    A = (np.abs(r_full) > threshold).astype(float)
    np.fill_diagonal(A, 0)
    G = nx.from_numpy_array(A)
    im = infomap.Infomap("--regularized --two-level --silent --num-trials 5")
    im.add_networkx_graph(G)
    im.run()
    reg_codelength_savings.append(im.relative_codelength_savings)

threshold_best_reg = thresholds[np.argmax(reg_codelength_savings)]
print(f"Best threshold (cross-validation):    {threshold_best:.2f}")
print(f"Best threshold (Regularized Map Eq.): {threshold_best_reg:.2f}")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(3.5, 3.5))
ax.plot(
    thresholds,
    train_codelength_savings[0] / nRuns,
    "--",
    color="steelblue",
    label="CV training",
)
ax.plot(
    thresholds, test_codelength_savings[0] / nRuns, color="steelblue", label="CV test"
)
ax.plot(
    thresholds, reg_codelength_savings, color="darkorange", label="Regularized Map Eq."
)
ax.plot(
    threshold_best,
    test_codelength_savings.max() / nRuns,
    "o",
    color="steelblue",
    label=f"CV best τ = {threshold_best:.2f}",
)
ax.plot(
    threshold_best_reg,
    max(reg_codelength_savings),
    "o",
    color="darkorange",
    label=f"Reg. best τ = {threshold_best_reg:.2f}",
)
ax.set_xlim(0.0, 0.72)
ax.legend(fontsize=9, frameon=False)
ax.set_xlabel("Threshold")
ax.set_ylabel("Description length compression ΔD")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

In [ ]:
# Compare the partitions produced by the two threshold-selection methods
# using the same spring layout so node positions are directly comparable
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

rng = np.random.default_rng(42)

# --- Regularized Map Equation (left panel) ---
# Binary (unweighted) network: RegMapEq operates on presence/absence of edges
A_reg = (np.abs(r_full) > threshold_best_reg).astype(float)
np.fill_diagonal(A_reg, 0)
G_reg = nx.from_numpy_array(A_reg)
im_reg = infomap.Infomap("--regularized --two-level --silent --num-trials 5")
im_reg.add_networkx_graph(G_reg)
im_reg.run()
modules_reg = dict(im_reg.modules)
num_modules_reg = im_reg.num_top_modules
AMI_reg = adjusted_mutual_info_score(list(modules_reg.values()), plantedPartition)

# Build shared layout from connected nodes; scatter isolates near the centre
# so the spring layout is not distorted by degree-0 nodes
connected = [n for n in G_reg.nodes if G_reg.degree(n) > 0]
pos = nx.forceatlas2_layout(
    G_best.subgraph(connected),
    jitter_tolerance=1.0,
    scaling_ratio=0.2,
    gravity=1.5,
    strong_gravity=True,
    node_mass=None,
    node_size=None,
    weight=None,
    seed=42,
)
center = np.mean(list(pos.values()), axis=0)
for n in nx.isolates(G_reg):
    pos[n] = center + rng.uniform(-0.1, 0.1, 2)

palette_reg = sns.color_palette("Set3", n_colors=num_modules_reg)
node_colors_reg = [palette_reg[modules_reg[n] - 1] for n in G_reg.nodes]
nx.draw(
    G_reg,
    pos=pos,
    ax=ax1,
    node_color=node_colors_reg,
    node_size=150,
    edgecolors="gray",
    linewidths=0.2,
    edge_color=make_edge_colors(G_reg, modules_reg, palette_reg, rng),
    width=0.5,
)
ax1.set_title(
    f"Regularized Map Eq.  τ* = {threshold_best_reg:.2f}\n{im_reg.num_non_trivial_top_modules} non-trivial modules, AMI = {AMI_reg:.2f}"
)
ax1.axis("off")

# --- Cross-validation (right panel) ---
# Weighted network: consistent with the threshold-selection cell above
r_abs = np.abs(np.dot(sample.T, sample) / sample.shape[0])
np.fill_diagonal(r_abs, 0)
G_cv = nx.from_numpy_array(r_abs * (r_abs > threshold_best))
im_cv = infomap.Infomap(silent=True, two_level=True)
im_cv.add_networkx_graph(G_cv)
im_cv.run()
modules_cv = dict(im_cv.modules)
num_modules_cv = im_cv.num_top_modules
AMI_cv = adjusted_mutual_info_score(list(modules_cv.values()), plantedPartition)

palette_cv = sns.color_palette("Set3", n_colors=num_modules_cv)
node_colors_cv = [palette_cv[modules_cv[n] - 1] for n in G_cv.nodes]
nx.draw(
    G_cv,
    pos=pos,
    ax=ax2,
    node_color=node_colors_cv,
    node_size=150,
    edgecolors="gray",
    linewidths=0.2,
    edge_color=make_edge_colors(G_cv, modules_cv, palette_cv, rng),
    width=0.5,
)
ax2.set_title(
    f"Cross-validation  τ* = {threshold_best:.2f}\n{im_cv.num_non_trivial_top_modules} non-trivial modules, AMI = {AMI_cv:.2f}"
)
ax2.axis("off");